### AIRCRAFT LANDING SCHDULING

##### *Problem Statement*

The first task involves scheduling aircraft landings on a single runway. 
The objective is to *minimize the total penalty cost for early and late landings* ensuring aircraft land within their specified time windows and *required minimum separation times between 2 landings are maintained*.

##### *Data*
The data consists of the following tables: <br>
**Table 1**: Aircraft Time Windows and Penalty Costs <br>
**Table 2**: Minimum Separation Times 

##### *Model*

1. **Variables**: <br>
$t_i$ = landing time of aircraft $i$ <br>
$e_i$ = early landing penalty of aircraft $i$ <br>
$l_i$ = late landing penalty of aircraft $i$ <br>
$x[i,j]$ = binary variable indicating if aircraft $i$ lands before aircraft $j$ <br>

2. **Objective Function**: <br>
Minimize the total penalty cost for early and late landings <br>
$Minimize \sum_{i} (penalty\_early_i . e_i + penalty\_late_i . l_i)$ 

3. **Constraints**: <br>
- Aircraft land within their specified time windows <br>
$t_i \geq Earliest\_landing_i$ <br>
$t_i \leq Latest\_landing_i$ <br>

- Penalty Calculation <br>
$e_i \geq max(0, Estimated_i - t_i)$ <br>
$l_i \geq max(0, t_i - Estimated_i)$

- Required minimum separation times between 2 landings are maintained <br>
$t_j \geq t_i + Separation_{i,j}\; i \not= j$




In [1]:
import gurobipy as gp
from gurobipy import GRB

# Data
aircraft = range(10)
earliest = [129, 195, 89, 96, 110, 120, 124, 126, 135, 160]
estimated = [155, 258, 96, 106, 123, 135, 138, 140, 150, 180]
latest = [559, 744, 510, 521, 555, 576, 577, 573, 591, 657]
penalty_early = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
penalty_late = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
separation = [
    [0, 3, 15, 15, 15, 15, 15, 15, 15, 15],
    [3, 0, 15, 15, 15, 15, 15, 15, 15, 15],
    [15, 15, 0, 8, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 0, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 0, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 0, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 0, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 0, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 0, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 8, 0]
]

# Create a new model
m = gp.Model("aircraft_landing")

# Add variables
t = m.addVars(aircraft, lb=earliest, ub=latest, name="t")
e = m.addVars(aircraft, vtype=GRB.CONTINUOUS, name="e")
l = m.addVars(aircraft, vtype=GRB.CONTINUOUS, name="l")
x = m.addVars(aircraft, aircraft, vtype=GRB.BINARY, name="x")

# Set objective
m.setObjective(gp.quicksum(penalty_early[i] * e[i] + penalty_late[i] * l[i] for i in aircraft), GRB.MINIMIZE)

# Add constraints for time windows and penalties
for i in aircraft:
    m.addConstr(e[i] >= estimated[i] - t[i], name=f"early_penalty_{i}")
    m.addConstr(l[i] >= t[i] - estimated[i], name=f"late_penalty_{i}")

# Add constraints for separation times using big-M approach
big_M = 1000  # Adjust as necessary
for i in aircraft:
    for j in aircraft:
        if i != j:
            m.addConstr(t[j] >= t[i] + separation[i][j] - big_M * (1 - x[i, j]), name=f"separation_{i}_{j}_1")
            m.addConstr(t[i] >= t[j] + separation[i][j] - big_M * x[i, j], name=f"separation_{i}_{j}_2")
            m.addConstr(x[i, j] + x[j, i] == 1, name=f"order_{i}_{j}")

# Optimize model
m.optimize()

# Check if optimization was successful
if m.status == GRB.OPTIMAL:
    for i in aircraft:
        print(f"Aircraft {i+1}: Landing time = {t[i].X}, Early penalty = {e[i].X}, Late penalty = {l[i].X}")
else:
    print("Optimization was not successful.")
    if m.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
        m.computeIIS()
        m.write("model.ilp")
        for constr in m.getConstrs():
            if constr.IISConstr:
                print(f"Infeasible constraint: {constr.ConstrName}")
    elif m.status == GRB.UNBOUNDED:
        print("Model is unbounded.")
    else:
        print(f"Optimization ended with status {m.status}")


Restricted license - for non-production use only - expires 2025-11-24
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 290 rows, 130 columns and 760 nonzeros
Model fingerprint: 0xdb39efeb
Variable types: 30 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+01, 3e+01]
  Bounds range     [1e+00, 7e+02]
  RHS range        [1e+00, 1e+03]
Found heuristic solution: objective 8950.0000000
Presolve removed 190 rows and 55 columns
Presolve time: 0.02s
Presolved: 100 rows, 75 columns, 300 nonzeros
Variable types: 30 continuous, 45 integer (45 binary)

Root relaxation: objective 0.000000e+00, 41 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl

The best objective value (total penalty cost) obtained is $700$. which matches the best bound indicating the optimality of the solution (as indicted by the zero gap between the best objective and the best bound) <br>
We can also note that aircraft $5,6,7$ incur early penalty costs and aircraft $1$ and $8$ incur late penalty costs.<br>
The solution respects all constraints, including the minimum separation times between 2 landings therefore ensuring *feasibility* of the solution.


### OPTIMAL DELIVERY ROUTES FOR OIL

##### *Problem Statement*
In this task, the goal is to determine the delivery routes that minimize the total travel distance required to satisfy the demand for oil at various cities using a tankers with a capacty of 39000 liters.

##### *Model Formulation*
This problem can be formulated as a *Vehicle Routing Problem (VRP)* with a single depot (refinery $\Omega$) and multiple delivery points (cities A-F).

1. **Variables**: <br>
$x_{ij}$ = binary variable indicating if tanker travels from city $i$ to city $j$ <br>
$q_i$ = Continuous varible representing the remaining capacity of the tanker after delivering to city $i$ <br>

2. **Objective Function**: <br>
Minimize the total travel distance <br>
$Minimize \sum_{i}\sum_{j} d_{ij} . x_{ij}$

3. **Constraints**: <br>
- Flow Conservation <br>
Ensure that each city is visited exactly once <br>
$\sum_{j} x_{ij} = 1 \; \forall i \in \{A,B,C,D,E,F\}$ <br>
$\sum_{i} x_{ij} = 1 \; \forall j \in \{A,B,C,D,E,F\}$ <br>

- Capacity Constraints <br>
Ensure that the tanker does not exceed its capacity <br>
$q_j \leq q_i - demand_j + (1 - x_{ij}) . 39000 \; \forall i,j$ <br>

- Depot Constraints <br>
Ensure that the tanker returns to the depot after visiting all cities <br>
$\sum_{j} x_{\Omega j} = 1$ <br>
$\sum_{i} x_{i\Omega} = 1$ <br>

- Subtour Elimination <br>
Eliminate subtours by adding subtour elimination constraints <br>
$u_i - u_j + Q . x_{ij} \leq Q - demand_j \; \forall i,j,i \not= j$ <br>

In [1]:
import gurobipy as gp
from gurobipy import GRB

# Data
cities = ['Ω', 'A', 'B', 'C', 'D', 'E', 'F']
demand = [0, 14000, 3000, 6000, 16000, 15000, 5000]  # Including 'depot' with 0 demand
distance_matrix = [
    [0, 148, 55, 32, 70, 140, 73],
    [148, 0, 93, 180, 99, 12, 72],
    [55, 93, 0, 85, 20, 83, 28],
    [32, 180, 85, 0, 100, 174, 99],
    [70, 99, 20, 100, 0, 85, 49],
    [140, 12, 83, 174, 85, 0, 73],
    [73, 72, 28, 99, 49, 73, 0]
]
tanker_capacity = 39000

num_cities = len(cities)

# Model
m = gp.Model("oil_delivery")

# Variables
x = m.addVars(num_cities, num_cities, vtype=GRB.BINARY, name="x")
u = m.addVars(num_cities, vtype=GRB.CONTINUOUS, name="u")

# Objective
m.setObjective(gp.quicksum(distance_matrix[i][j] * x[i, j] for i in range(num_cities) for j in range(num_cities)), GRB.MINIMIZE)

# Constraints
# Flow conservation constraints
for i in range(num_cities):
    m.addConstr(gp.quicksum(x[i, j] for j in range(num_cities) if i != j) == 1, name=f"flow_conservation_out_{i}")
    m.addConstr(gp.quicksum(x[j, i] for j in range(num_cities) if i != j) == 1, name=f"flow_conservation_in_{i}")

# Depot constraints
m.addConstr(gp.quicksum(x[0, j] for j in range(1, num_cities)) == 1, name="depot_out")
m.addConstr(gp.quicksum(x[i, 0] for i in range(1, num_cities)) == 1, name="depot_in")

# MTZ Subtour elimination constraints
for i in range(1, num_cities):
    m.addConstr(u[i] >= 1, name=f"mtz_lower_bound_{i}")
    m.addConstr(u[i] <= num_cities - 1, name=f"mtz_upper_bound_{i}")

for i in range(1, num_cities):
    for j in range(1, num_cities):
        if i != j:
            m.addConstr(u[i] - u[j] + (num_cities - 1) * x[i, j] <= num_cities - 2, name=f"mtz_{i}_{j}")

# Optimize model
m.optimize()

# Check if optimization was successful
if m.status == GRB.OPTIMAL:
    for i in range(num_cities):
        for j in range(num_cities):
            if x[i, j].X > 0.5:
                print(f"Route from {cities[i]} to {cities[j]}")
else:
    print("Optimization was not successful.")
    if m.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
        m.computeIIS()
        m.write("model.ilp")
        for constr in m.getConstrs():
            if constr.IISConstr:
                print(f"Infeasible constraint: {constr.ConstrName}")
    elif m.status == GRB.UNBOUNDED:
        print("Model is unbounded.")
    else:
        print(f"Optimization ended with status {m.status}")


Restricted license - for non-production use only - expires 2025-11-24
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 58 rows, 56 columns and 198 nonzeros
Model fingerprint: 0x98ab5ed9
Variable types: 7 continuous, 49 integer (49 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+00]
  Objective range  [1e+01, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]
Presolve removed 14 rows and 8 columns
Presolve time: 0.00s
Presolved: 44 rows, 48 columns, 294 nonzeros
Variable types: 6 continuous, 42 integer (42 binary)
Found heuristic solution: objective 379.0000000

Root relaxation: objective 2.154000e+02, 15 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl | 

**Note**: I used the Miller-Tucker-Zemlin (MTZ) formulation to prevent subtours ensure the single solution forms a single and continuous route. :
- The tanker starts from the depot $\Omega$ and goes to city $B$.<br>
- From city $B$, the tanker moves to city $D$, and then to city $E$.<br>
- From city $E$, the tanker travels to city $A$, and then to city $F$. <br>
- From city $F$, the tanker goes to city $C$, and finally returns to the depot $\Omega$.<br>

The optimal solution obtained has a total travel distance of $375\; km$ which is the best possible solution. The solution respects all constraints including the capacity constraints and subtour elimination constraints, therefore ensuring *feasibility* of the solution